# Small and Specialized Models: When Not to Use a Frontier LLM

The runnable code of [Small and Specialized Models: When Not to Use a Frontier LLM](https://residual-stream.lvtan71.workers.dev/posts/small-and-specialized-models/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/small-and-specialized-models.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 50 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "small-and-specialized-models"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/small-and-specialized-models/lm.py', 'shared/small-and-specialized-models/quiet_machine.py', 'shared/softmax-and-cross-entropy/router4.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/softmax-and-cross-entropy/softmax_basics.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## What "small" buys

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/small-and-specialized-models/#what-small-buys)

In [ ]:
# What "small" buys. Decoding one token reads every weight once, so with a
# batch of one, generation speed is bounded by memory bandwidth: tokens per
# second <= bandwidth / bytes of weights ("How a GPU Runs Deep Learning"'s roofline, memory-bound).
# Bandwidths: this laptop's GPU as measured in "How a GPU Runs Deep Learning" (93 GB/s), and an
# H100's 3.35 TB/s. Memory: the laptop's 16 GB, shared with everything else,
# and the H100's 80 GB; "-" marks weights that don't fit. Reading the KV
# cache ("The KV Cache") is left out: small at short context, not at long context.
LAPTOP, H100 = (93e9, 16e9), (3.35e12, 80e9)
models = [("0.5B", 0.5e9), ("1.5B", 1.5e9), ("3B", 3e9), ("8B", 8e9), ("70B", 70e9)]

def cap(device, size):
    bandwidth, memory = device
    return f"{bandwidth / size:5.0f}" if size < memory else "    -"

print(f"{'':6s} {'BF16':>8s} {'INT4':>8s}   tokens/s at most, batch 1:  laptop BF16 / INT4    H100 BF16 / INT4")
for name, n in models:
    bf16, int4 = 2 * n, 0.5 * n
    print(f"{name:6s} {bf16 / 1e9:6.1f}GB {int4 / 1e9:6.2f}GB   {'':26s}  {cap(LAPTOP, bf16)} / {cap(LAPTOP, int4)}    "
          f"{cap(H100, bf16)} / {cap(H100, int4)}")
# A check against a measurement: "Decoding" timed Qwen2.5-1.5B (1.54B
# parameters, 16-bit) at 41 ms per token on this laptop.
n = 1.54e9
print(f"\nQwen2.5-1.5B in 16 bits: {2 * n / 1e9:.2f} GB per token read; at the measured 41 ms per token that's "
      f"{2 * n / 0.041 / 1e9:.0f} GB/s, {2 * n / 0.041 / LAPTOP[0]:.0%} of the measured bandwidth")

The post printed:

```text
           BF16     INT4   tokens/s at most, batch 1:  laptop BF16 / INT4    H100 BF16 / INT4
0.5B      1.0GB   0.25GB                                  93 /   372     3350 / 13400
1.5B      3.0GB   0.75GB                                  31 /   124     1117 /  4467
3B        6.0GB   1.50GB                                  16 /    62      558 /  2233
8B       16.0GB   4.00GB                                   - /    23      209 /   838
70B     140.0GB  35.00GB                                   - /     -        - /    96

Qwen2.5-1.5B in 16 bits: 3.08 GB per token read; at the measured 41 ms per token that's 75 GB/s, 81% of the measured bandwidth
```

## An experiment: routing the notebook's questions

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/small-and-specialized-models/#an-experiment-routing-the-notebooks-questions)

In [ ]:
import re
import time
import quiet_machine                                   # waits for an idle machine before anything is timed
import numpy as np
import torch
from router_data import split, ROUTES, english, tokens
from router4 import four_way, train, top_label_ece
from softmax_basics import softmax

# Ways to make one typed decision, which of four routes a question belongs
# to, on the series' 200 English router questions. Every question gets one
# out-of-sample prediction: the trained models use the first of 40 random
# splits (the 120 / 40 / 40 split of "Features and Overfitting") in which that question is a test
# question; the language model is not trained, and answers each question
# once (with examples, they come from that same split's training set).
#   1. the bag-of-words softmax router of "Cross-Entropy and the Softmax Gradient";
#   2. sentence embeddings (all-MiniLM-L6-v2, 23M parameters, "Embeddings and Contrastive Learning") with
#      the same softmax head, and with no training at all: nearest centroid;
#   3. Qwen2.5-1.5B-Instruct, untrained, zero-shot and with 20 labeled
#      examples in the prompt, read two ways: typed, by the probability of
#      every way of spelling each route name; or free, generating and parsing.
SPLITS = [split(seed) for seed in range(40)]
home = {}                                                   # question -> index of its test split
for k, (_, _, test) in enumerate(SPLITS):
    for r in test:
        home.setdefault(r["question"], k)
questions = [r["question"] for r in english]
truth = np.array([ROUTES.index(r["route"]) for r in english])
assert len(home) == 200
preds = {}

# 1. Bag of words, and its batch-1 latency, text in to probabilities out.
bow, t_bow = {}, []
for k, (tr, _, te) in enumerate(SPLITS):
    ((Xtr, ytr), _), vocab = four_way(tr, te)
    W, b = train(Xtr, ytr, 0.0, 0.1)                                       # the choice of "Cross-Entropy and the Softmax Gradient"
    col = {t: j for j, t in enumerate(vocab)}
    for r in te:
        if home[r["question"]] != k:
            continue
        t0 = time.perf_counter()
        x = np.zeros(len(vocab))
        for t in set(tokens(r["question"])):
            if t in col:
                x[col[t]] = 1
        p = softmax(x[None] @ W + b)[0]
        t_bow.append(time.perf_counter() - t0)
        bow[r["question"]] = p
preds["bag-of-words router"] = (bow, np.median(t_bow))

# 2. Sentence embeddings.
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
E = dict(zip(questions, enc.encode(questions, normalize_embeddings=True)))
t_enc = []
for q in questions[:50]:                                                   # batch-1 encode time
    t0 = time.perf_counter(); enc.encode([q], normalize_embeddings=True); t_enc.append(time.perf_counter() - t0)
head, centroid = {}, {}
for k, (tr, _, te) in enumerate(SPLITS):
    Xtr = np.array([E[r["question"]] for r in tr]); ytr = np.array([ROUTES.index(r["route"]) for r in tr])
    W, b = train(Xtr * 8, ytr, 0.0, 0.1)                                   # x8: unit vectors train faster scaled up
    C = np.array([Xtr[ytr == c].mean(0) for c in range(4)])
    for r in te:
        if home[r["question"]] == k:
            e = E[r["question"]]
            head[r["question"]] = softmax((e * 8)[None] @ W + b)[0]
            centroid[r["question"]] = softmax((e @ C.T)[None] * 20)[0]    # cosine to each route's mean
preds["MiniLM + softmax head"] = (head, np.median(t_enc))
preds["MiniLM, nearest centroid"] = (centroid, np.median(t_enc))

# 3. The language model.
from lm import model, tok, DEVICE
m = model()
DESCRIBE = ("serving: running models in production (KV cache, batching, vLLM, latency); "
            "architecture: how models are built (attention, layers, positions, MoE); "
            "agents: tools, planning and agent loops; "
            "foundations: training, losses, optimization and math.")
SPELLINGS = {r: [r, r.capitalize(), " " + r, " " + r.capitalize()] for r in ROUTES}
SPELLINGS["foundations"] += ["foundation", "Foundation", " foundation", " Foundation"]
SPELLINGS["agents"] += ["agent", "Agent", " agent", " Agent"]
cands = [(ROUTES.index(r), tok(s).input_ids) for r, forms in SPELLINGS.items() for s in forms]

def prompt(q, shots=None):
    if shots is None:
        text = (f"Classify the question into exactly one route.\nRoutes: {DESCRIBE}\nQuestion: {q}\n"
                "Answer with one word: serving, architecture, agents or foundations.")
    else:
        examples = "\n".join(f"Question: {r['question']}\nRoute: {r['route']}" for r in shots)
        text = (f"Classify the question into exactly one route.\nRoutes: {DESCRIBE}\n"
                f"Labeled examples:\n{examples}\n\nQuestion: {q}\n"
                "Answer with one word: serving, architecture, agents or foundations.")
    return tok.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True,
                                   return_tensors="pt", return_dict=True)["input_ids"][0].tolist()

@torch.no_grad()
def typed(ids):
    """Probability of each route: the summed probability of every spelling's
    full token sequence after the prompt, scored in one batched forward."""
    L = max(len(c) for _, c in cands)
    seqs = [ids + c + [tok.eos_token_id] * (L - len(c)) for _, c in cands]
    h = m.model(torch.tensor(seqs, device=DEVICE)).last_hidden_state[:, len(ids) - 1:]   # only the scored positions
    logp = m.lm_head(h).float().log_softmax(-1)
    p = np.zeros(4)
    for i, (route, c) in enumerate(cands):
        p[route] += float(sum(logp[i, j, t] for j, t in enumerate(c)).exp())
    return p / p.sum(), p.sum()

def parse(text):
    """A forgiving parser: lowercase, drop punctuation and markdown, accept
    the singular and a leading prefix."""
    w = re.sub(r"[^a-z ]", " ", text.lower()).split()
    for word in w[:3]:
        for r in ROUTES:
            if word in (r, r.rstrip("s")) or (len(word) >= 5 and r.startswith(word)):
                return ROUTES.index(r)
    return None

def shots_for(q):
    tr = SPLITS[home[q]][0]
    rng = np.random.default_rng(home[q])
    pool = [tr[i] for i in rng.permutation(len(tr))]
    s = [r for route in ROUTES for r in [x for x in pool if x["route"] == route][:5]]
    rng.shuffle(s)
    return s

for name, with_shots in (("Qwen2.5-1.5B zero-shot", False), ("Qwen2.5-1.5B, 20 examples", True)):
    typed_p, free_p, covered, invalid, t_pre, t_gen, lens = {}, {}, [], 0, [], [], []
    for q in questions:
        ids = prompt(q, shots_for(q) if with_shots else None)
        lens.append(len(ids))
        x = torch.tensor([ids], device=DEVICE)
        torch.mps.synchronize(); t0 = time.perf_counter()
        with torch.no_grad():
            m(x)
        torch.mps.synchronize(); t_pre.append(time.perf_counter() - t0)
        typed_p[q], mass = typed(ids); covered.append(mass)
        torch.mps.synchronize(); t0 = time.perf_counter()
        out = m.generate(x, max_new_tokens=8, do_sample=False, repetition_penalty=1.0)
        torch.mps.synchronize(); t_gen.append(time.perf_counter() - t0)
        r = parse(tok.decode(out[0, len(ids):], skip_special_tokens=True))
        if r is None:
            invalid += 1
        free_p[q] = None if r is None else np.eye(4)[r]
    preds[f"{name}, typed"] = (typed_p, np.median(t_pre))
    preds[f"{name}, free"] = (free_p, np.median(t_gen))
    print(f"{name}: prompts of {np.mean(lens):.0f} tokens on average; the route spellings carry "
          f"{np.median(covered):.0%} of the first-word probability (median); free answers unparseable: {invalid} of 200")

# Scores, one prediction per question. 95% intervals by bootstrap over questions.
rng = np.random.default_rng(0)
boots = [rng.integers(0, 200, 200) for _ in range(2000)]
print(f"\n{'':40s} {'accuracy':>16s} {'ECE':>6s} {'batch-1 time':>13s}")
correct = {}
for name, (p, t) in preds.items():
    right = np.array([p[q] is not None and p[q].argmax() == y for q, y in zip(questions, truth)], dtype=float)
    correct[name] = right
    lo, hi = np.percentile([right[i].mean() for i in boots], [2.5, 97.5])
    if all(p[q] is not None for q in questions) and "free" not in name:
        P = np.array([p[q] for q in questions]); ece = f"{top_label_ece(P, truth):6.3f}"
    else:
        ece = "   n/a"                                                       # generated labels carry no confidence
    when = "" if t is None else (f"{t * 1e6:8.1f} us" if t < 1e-3 else f"{t * 1e3:8.1f} ms")
    print(f"{name:40s} {right.mean():.3f} [{lo:.2f}, {hi:.2f}] {ece} {when:>13s}")
d = correct["MiniLM + softmax head"] - correct["Qwen2.5-1.5B, 20 examples, typed"]
lo, hi = np.percentile([d[i].mean() for i in boots], [2.5, 97.5])
print(f"\nMiniLM head minus Qwen with examples (typed): {d.mean():+.3f} [{lo:+.2f}, {hi:+.2f}]")
P = np.array([head[q] for q in questions])
for th in (0.6, 0.7, 0.8):
    keep = P.max(1) >= th
    print(f"MiniLM head, answering only at confidence >= {th}: {keep.mean():.0%} of questions answered, "
          f"{(P[keep].argmax(1) == truth[keep]).mean():.1%} of those right")
qp = np.array([preds['Qwen2.5-1.5B, 20 examples, typed'][0][q] for q in questions])
print("Qwen with examples, typed, right by true route: " + ", ".join(
    f"{r} {np.mean(qp[truth == i].argmax(1) == i):.2f}" for i, r in enumerate(ROUTES)))

The post printed:

```text
Qwen2.5-1.5B zero-shot: prompts of 118 tokens on average; the route spellings carry 99% of the first-word probability (median); free answers unparseable: 0 of 200
Qwen2.5-1.5B, 20 examples: prompts of 439 tokens on average; the route spellings carry 98% of the first-word probability (median); free answers unparseable: 0 of 200

                                                 accuracy    ECE  batch-1 time
bag-of-words router                      0.655 [0.59, 0.71]  0.050        6.0 us
MiniLM + softmax head                    0.710 [0.65, 0.77]  0.035        2.6 ms
MiniLM, nearest centroid                 0.780 [0.72, 0.83]  0.158        2.6 ms
Qwen2.5-1.5B zero-shot, typed            0.540 [0.47, 0.61]  0.291      155.9 ms
Qwen2.5-1.5B zero-shot, free             0.525 [0.46, 0.59]    n/a      215.8 ms
Qwen2.5-1.5B, 20 examples, typed         0.565 [0.50, 0.64]  0.274      465.6 ms
Qwen2.5-1.5B, 20 examples, free          0.555 [0.49, 0.62]    n/a      521.4 ms

MiniLM head minus Qwen with examples (typed): +0.145 [+0.06, +0.23]
MiniLM head, answering only at confidence >= 0.6: 70% of questions answered, 82.7% of those right
MiniLM head, answering only at confidence >= 0.7: 57% of questions answered, 88.6% of those right
MiniLM head, answering only at confidence >= 0.8: 41% of questions answered, 92.7% of those right
Qwen with examples, typed, right by true route: serving 0.92, architecture 0.96, agents 0.12, foundations 0.26
```